# CO₂ Blowdown with Real-Fluid Thermodynamics (Span–Wagner)

Single-phase gaseous CO₂ blowdown of the LSU well with the Span–Wagner
reference EOS (via CoolProp), compared against N₂ at matched conditions
(5 MPa wellhead / 300 K, hydrostatic initialization). The solver monitors
the saturation margin every step and stops with a flagged event on first
dome entry. Details: `docs/co2_eos.md`.

In [ ]:
try:
    import hyperbolic1d, CoolProp
except ImportError:
    %pip install -q CoolProp "hyperbolic1d[plot] @ git+https://github.com/JaderBarbosaLSU/hyperco2-wellbore"
import numpy as np, matplotlib.pyplot as plt
from hyperbolic1d.realfluid import SpanWagnerCO2
from hyperbolic1d.eos import NITROGEN
from hyperbolic1d.wellbore import WellboreModel, WellGeometry, ChokeValve
from hyperbolic1d.thermal import lsu_co2_well_stack, FormationCoupledWall

## 1. EOS sanity: how non-ideal is the starting state?

In [ ]:
eos = SpanWagnerCO2()
rho, e = eos.rho_e_from_pT(5e6, 300.0)
p, T, a = eos.primitives(np.array([rho]), np.array([e]))
print(f"CO2 @ 5 MPa / 300 K: rho = {rho:.1f} kg/m3 (ideal: {5e6/(188.92*300):.1f}),"
      f" a = {a[0]:.0f} m/s (ideal: {np.sqrt(1.289*188.92*300):.0f})")
print("table accuracy vs direct Span-Wagner:", eos.table_verification(100))

## 2. Matched N₂ vs CO₂ runs (coupled formation model)

In [ ]:
r1, *_ = lsu_co2_well_stack()
geom  = WellGeometry(depth=609.6, diameter=2*r1)
choke = ChokeValve()
T_END = 1800.0   # 3600 s used for the archived campaign
runs = {}
for lab, eos_i in (("CO$_2$", SpanWagnerCO2()), ("N$_2$", NITROGEN)):
    m = WellboreModel(geometry=geom, choke=choke, eos=eos_i, gravity=True,
                      friction=True, n_cells=200,
                      wall_model=FormationCoupledWall())
    runs[lab] = m.solve(5e6, 300.0, T_END, init="hydrostatic", n_snapshots=5)
    r = runs[lab]
    tag = (f"TWO-PHASE ENTRY @ {r.two_phase_entry['t']:.0f}s"
           if r.two_phase_entry else
           f"single-phase (min margin {r.sat_margin_min.min():.1f} K)"
           if r.sat_margin_min is not None else "")
    print(f"{lab}: mdot0={r.mdot[0]:.2f} kg/s, M0={r.mass[0]:.0f} kg, "
          f"T_wh min={r.T_wellhead.min():.1f} K  {tag}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4))
for lab, r in runs.items():
    c = "tab:red" if "CO" in lab else "tab:blue"
    axes[0].plot(r.t_hist/60, r.p_wellhead/1e6, c=c, label=lab)
    axes[1].plot(r.t_hist/60, r.mdot, c=c)
    axes[2].plot(r.t_hist/60, r.mass/r.mass[0], c=c)
for ax, lab in zip(axes, ["$p_{wh}$ [MPa]", "$\\dot m$ [kg/s]", "$M/M_0$"]):
    ax.set_xlabel("t [min]"); ax.set_ylabel(lab); ax.grid(alpha=0.3)
axes[0].legend(); fig.tight_layout()
# CO2 holds ~2.3x the mass at equal pressure (Z=0.69, heavier molecule)
# and discharges a comparable mdot -> much longer blowdown.

## 3. Saturation-margin trajectory

In [ ]:
r = runs["CO$_2$"]
eos._ensure_sat_tables()
fig, ax = plt.subplots(figsize=(6,4.5))
ax.plot(eos._T_sat, np.exp(eos._lnp_sat)/1e6, "k-", label="saturation curve")
ax.plot(eos.T_crit, eos.p_crit/1e6, "k*", ms=9)
ax.plot(r.T[:, -1], r.p[:, -1]/1e6, "r-", lw=1, label="wellhead cell")
ax.plot(r.T[:, 0],  r.p[:, 0]/1e6,  "r--", lw=1, label="bottom cell")
ax.set_xlabel("T [K]"); ax.set_ylabel("p [MPa]"); ax.grid(alpha=0.3); ax.legend()
ax.set_title("CO$_2$ p-T trajectory vs vapor dome");

## 4. Experiments

* `init="uniform"` with `p0=6.5e6`: the settling transient crosses the
  dome within a second — the solver stops and reports the entry state.
* Adiabatic wall (`wall_model` default): watch the margin shrink as
  expansion cooling proceeds unopposed.
* `ChokeValve(diameter=0.0254)`: faster discharge, stronger cooling.